In [8]:
import os
import rasterio
from importlib_resources import files
from pathlib import Path
from rasterio.enums import Resampling
from tqdm import tqdm

from beak.utilities.raster_processing import unify_raster_grids
from beak.utilities.io import save_raster, create_file_list, check_path, create_file_folder_list


**User inputs**
1. Select the root folder where the rasters are stored.
2. Then, go down to select the subfolders that contain the rasters to be unified.<p>

The script will search for all rasters and store them in relative paths.

Original **YTU** were just copied from the RAW folder to the PROCESSED folder, since they are the blueprint for the base raster and thus do not need to be unified.

In [9]:
BASE_PATH = files("beak.data")
BASE_SPATIAL = "102008_500"
BASE_EXTENT = "mvt_ce"
BASE_RASTER = BASE_PATH / "BASE_RASTERS" / "EPSG_102008_RES_500_MVT_CE.tif"
BASE_EVIDENCE = "geophysics"

# Export
# Some data sets are already **LOG** scaled and need special actions. They will be unified and stored in a separate folder.
PATH_INPUT = BASE_PATH / "RAW" / BASE_EVIDENCE / "national_scale"

PATH_ROOT = BASE_PATH / "PROCESSED" / str("regional" + "_" + BASE_SPATIAL + "_" + BASE_EXTENT)
PATH_EXPORT = PATH_ROOT / "unified" / BASE_EVIDENCE
PATH_EXPORT_LOG = PATH_ROOT / "unified_scaled_log" / BASE_EVIDENCE

CURRENT_DIR = Path(os.getcwd()) / PATH_EXPORT.name
OUT_FOLDER = PATH_EXPORT
OUT_FOLDER_LOG = PATH_EXPORT_LOG

print(f"Input folder: {PATH_INPUT}")
print(f"Export folder: {OUT_FOLDER}")

Input folder: s:\projekte\20230082_darpa_criticalmaas_ta3\bearbeitung\github\beak-ta3\src\beak\data\RAW\geophysics\national_scale
Export folder: s:\projekte\20230082_darpa_criticalmaas_ta3\bearbeitung\github\beak-ta3\src\beak\data\PROCESSED\regional_102008_500_mvt_ce\unified\geophysics


Select subfolders to be scaled.

In [10]:
for folder in os.listdir(PATH_INPUT):
  if os.path.isdir(os.path.join(PATH_INPUT, folder)):
    print(folder)


gravity
magnetic
magnetotelluric
radiometric
seismic
unsorted


In [11]:
SELECTION = ["gravity", 
             "magnetic",
             "magnetotelluric",
             "radiometric",
             "seismic"]

input_folders = [PATH_INPUT / folder for folder in SELECTION]

print("Selected folders:")
for folder in input_folders:
  print(folder)

Selected folders:
s:\projekte\20230082_darpa_criticalmaas_ta3\bearbeitung\github\beak-ta3\src\beak\data\RAW\geophysics\national_scale\gravity
s:\projekte\20230082_darpa_criticalmaas_ta3\bearbeitung\github\beak-ta3\src\beak\data\RAW\geophysics\national_scale\magnetic
s:\projekte\20230082_darpa_criticalmaas_ta3\bearbeitung\github\beak-ta3\src\beak\data\RAW\geophysics\national_scale\magnetotelluric
s:\projekte\20230082_darpa_criticalmaas_ta3\bearbeitung\github\beak-ta3\src\beak\data\RAW\geophysics\national_scale\radiometric
s:\projekte\20230082_darpa_criticalmaas_ta3\bearbeitung\github\beak-ta3\src\beak\data\RAW\geophysics\national_scale\seismic


**Select files**

In [12]:
# Create file list
file_list = []
for folder in input_folders:
  files = create_file_list(folder, recursive=True)
  file_list.extend(files)

# Remove CONUS 2021 data
file_list = [file for file in file_list if "CONUS_2021" not in str(file)]

# Create file list for log scaled data
log_files = [file.stem for file in file_list if "Log" in file.stem]

# Show results
print(f"Found {len(file_list)} files including {len(log_files)} log scaled files.")


Found 24 files including 1 log scaled files.


**Run unification**

In [13]:
# Select to write output file
dry_run = False

if dry_run:
  print("Dry run, no files will be written.\n")

# Unify
for file in tqdm(file_list, total=len(file_list)):
    folder_relative = os.path.relpath(file.parent, PATH_INPUT)

    raster = rasterio.open(file)
    unified_raster, unified_meta = unify_raster_grids(BASE_RASTER, [file], resampling_method=Resampling.bilinear, same_extent=True, same_shape=True)[0]

    if not dry_run:
      if file.stem in log_files:
        log_folder = OUT_FOLDER_LOG / str(folder_relative)
        log_out_path = log_folder / file.name
        check_path(Path(os.path.dirname(log_out_path)))
        save_raster(log_out_path, array=unified_raster, dtype="float32", metadata=unified_meta)
      else:
        output_folder = OUT_FOLDER / str(folder_relative)
        out_path = output_folder / file.name
        check_path(Path(os.path.dirname(out_path)))
        save_raster(out_path, array=unified_raster, dtype="float32", metadata=unified_meta)


  4%|▍         | 1/24 [00:02<00:47,  2.07s/it]

File already exists: Gravity.tif.


  8%|▊         | 2/24 [00:04<00:46,  2.11s/it]

File already exists: Gravity_HGM.tif.


 12%|█▎        | 3/24 [00:06<00:44,  2.10s/it]

...

File already exists: Mag_RTP_HGM.tif.


 54%|█████▍    | 13/24 [00:30<00:29,  2.65s/it]

File already exists: Mag_RTP_HGM_DeepSources.tif.


 58%|█████▊    | 14/24 [00:32<00:24,  2.47s/it]

File already exists: CONUS_MT2023_15km.tif.
